# Module 08 — Classes and Encapsulation

## Exercise 08.1 — The attribute lookup ladder

Twelve predictions. Write your answer AND the rung number (1-5) that the lookup
stopped at, before running.
    1 data descriptor on the type   2 instance __dict__
    3 the type and its bases        4 __getattr__        5 AttributeError
Run:  python ex01_lookup_lab.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. The attribute-lookup ladder

**The single most useful diagram in Part 2.** When you write `obj.x`, Python:

```text
1. type(obj).__mro__  -- looking for a DATA DESCRIPTOR named x
                         (something with __get__ AND __set__ -- e.g. @property)
                         found? call its __get__ and STOP.
2. obj.__dict__['x']  -- the instance's own dictionary
                         found? return it and STOP.
3. type(obj).__mro__  -- the class and its bases, in MRO order
                         found? return it (binding it if it is a function)
4. type(obj).__getattr__('x')   -- last-resort hook, if defined
5. AttributeError
```


Two consequences that explain a great deal:

**Instance attributes shadow class attributes** (step 2 beats step 3) — but
**properties beat instance attributes** (step 1 beats step 2). That ordering is
what makes `@property` able to intercept an attribute that used to be plain
data.

**A method is found on the class, not the instance.** Every instance of a class
shares one function object; the binding happens at lookup time.

In [ ]:
class Dog:
    def speak(self): return "woof"

d = Dog()
Dog.speak            # <function Dog.speak>       -- a plain function
d.speak              # <bound method Dog.speak>   -- function + instance
d.speak()            # == Dog.speak(d)

That is all `self` is: the first parameter, filled in by the binding. Python
makes it explicit rather than implicit, which is why you can do this:

In [ ]:
Dog.speak(d)                      # call it unbound
handler = d.speak                 # store a bound method as a callback
list(map(str.upper, ["a", "b"]))  # use an unbound method as a function

---

## Concept 4. There is no `private`

In [ ]:
class Account:
    def __init__(self):
        self.balance = 0          # public: part of the API
        self._ledger = []         # "internal": convention only
        self.__secret = "x"       # name-mangled, not private

| Form | Meaning | Enforced? |
|---|---|---|
| `name` | Public API | — |
| `_name` | Internal. Do not touch from outside. | No. Convention only. |
| `__name` | Mangled to `_ClassName__name` | Not privacy — see below |

`_name` is a message to other developers, and tools respect it: `from x import *`
skips it, IDEs de-emphasise it, documentation generators hide it. Nothing stops
you reading it. Python's position is that you are an adult and sometimes need to
reach into internals, and that a language-enforced barrier costs more than it
saves.

`__name` is different and frequently misunderstood. It exists for **one
purpose**: preventing accidental name collisions with subclasses.

In [ ]:
class Base:
    def __init__(self):
        self.__data = "base"      # becomes self._Base__data

class Child(Base):
    def __init__(self):
        super().__init__()
        self.__data = "child"     # becomes self._Child__data -- no collision

c = Child()
c._Base__data, c._Child__data     # ('base', 'child')  -- both alive

Use `__name` when you are writing a base class intended for subclassing and an
attribute must not be accidentally overridden. That is rare. Use `_name`
everywhere else.

---

## Concept 5. `@property`: why Python has no getters

In Java you write getters from the start because changing a public field to a
method later breaks every caller. **In Python it does not**, because
`@property` intercepts attribute access at the same syntax.

In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius      # start plain. No getter, no setter.

    @property
    def area(self) -> float:      # a computed, read-only attribute
        return 3.14159 * self.radius ** 2

c = Circle(2)
c.area                            # 12.56...   -- no parentheses
c.area = 5                        # AttributeError: property has no setter

Adding validation later, without changing any call site:

In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius      # this now goes through the setter

    @property
    def radius(self) -> float:
        return self._radius

    @radius.setter
    def radius(self, value: float) -> None:
        if value <= 0:
            raise ValueError(f"radius must be positive, got {value}")
        self._radius = value

Every existing `c.radius` and `c.radius = 5` keeps working, now validated. This
is why **you should not write a getter and setter until you need one.** Start
with a plain attribute; promote it to a property when there is a reason.

Two things to watch:

**Infinite recursion.** Inside the property, use `self._radius`, never
`self.radius` — the latter calls the property again.

**Cheapness.** A property looks like an attribute, so callers assume it is
cheap. A property that issues a database query will be called in a loop by
someone who had no way to know. If it is expensive, make it a method named
`compute_x()`, or cache it:

In [ ]:
from functools import cached_property

class Dataset:
    @cached_property
    def stats(self) -> dict[str, float]:      # computed once, then stored
        return expensive_analysis(self.rows)  # in the instance __dict__

`cached_property` works by writing the result into `self.__dict__`, so step 2 of
the lookup ladder finds it on every subsequent access and the descriptor never
runs again. (Which means it needs a `__dict__` — it does not work with
`__slots__`.)

---

## Concept 6. `@classmethod` and `@staticmethod`

In [ ]:
class Temperature:
    def __init__(self, kelvin: float) -> None:
        self.kelvin = kelvin

    @classmethod
    def from_celsius(cls, c: float) -> "Temperature":
        return cls(c + 273.15)             # cls, not Temperature

    @classmethod
    def from_fahrenheit(cls, f: float) -> "Temperature":
        return cls.from_celsius((f - 32) * 5 / 9)

    @staticmethod
    def is_valid_kelvin(value: float) -> bool:
        return value >= 0                   # no self, no cls

**`@classmethod` is how Python does named constructors.** A class can have only
one `__init__`, so alternative constructors become classmethods. Using `cls`
rather than the class name means subclasses get the right type back:

In [ ]:
class Kelvin(Temperature): ...
Kelvin.from_celsius(0)          # a Kelvin, not a Temperature

**`@staticmethod` is a function that lives in the class's namespace.** It gets
neither `self` nor `cls`. If it does not use either, ask whether it should be a
module-level function — often the honest answer is yes. It earns its place when
the grouping genuinely aids discovery, or when subclasses should be able to
override it.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: A class body is executable code
- Section 2: The attribute-lookup ladder
- Section 3: Class attributes versus instance attributes
- Section 4: There is no `private`
- Section 5: `@property`: why Python has no getters
- Section 6: `@classmethod` and `@staticmethod`
- Section 7: `__slots__`
- Section 8: Encapsulation that actually works

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from functools import cached_property

---

## `Base`

_Base_

In [ ]:
class Base:
    shared_list: list[str] = []
    shared_int = 0
    name = "base"

    def __init__(self) -> None:
        self.own = "instance"

    def method(self) -> str:
        return "from Base"

    @property
    def computed(self) -> str:
        return "computed"

    @cached_property
    def expensive(self) -> str:
        print("    (expensive ran)")
        return "cached"

---

## `q01`

_q01_

In [ ]:
def q01() -> None:
    # PREDICTION:                         rung:
    a, b = Base(), Base()
    a.shared_list.append("x")
    print("q01", b.shared_list)

---

## `q02`

_q02_

In [ ]:
def q02() -> None:
    # PREDICTION:                         rung:
    Base.shared_list = []   # reset -- q01 mutated the CLASS attribute, and it
                            # is still mutated. That cross-contamination between
                            # two functions that share no variables is itself
                            # the lesson of q01. Note how easily it hid here.
    a, b = Base(), Base()
    a.shared_list = ["x"]
    print("q02", b.shared_list, a.__dict__.get("shared_list"))

---

## `q03`

_q03_

In [ ]:
def q03() -> None:
    # PREDICTION:                         rung:
    a = Base()
    a.shared_int += 1
    print("q03", a.shared_int, Base.shared_int)

---

## `q04`

_q04_

In [ ]:
def q04() -> None:
    # PREDICTION:                         rung:
    a = Base()
    print("q04", a.name, a.__dict__.get("name"))

---

## `q05`

_q05_

In [ ]:
def q05() -> None:
    # PREDICTION:                         rung:
    a = Base()
    a.__dict__["computed"] = "sneaky"
    print("q05", a.computed, a.__dict__["computed"])

---

## `q06`

_q06_

In [ ]:
def q06() -> None:
    # PREDICTION:                         rung:
    a = Base()
    try:
        a.computed = "assigned"
        print("q06", a.computed)
    except Exception as exc:
        print("q06", type(exc).__name__)

---

## `q07`

_q07_

In [ ]:
def q07() -> None:
    # PREDICTION: how many times does "(expensive ran)" appear?
    a = Base()
    print("q07", a.expensive, a.expensive, a.expensive)
    print("q07 in __dict__?", "expensive" in a.__dict__)

---

## `q08`

_q08_

In [ ]:
def q08() -> None:
    # PREDICTION:                         rung:
    a = Base()
    print("q08", type(Base.method).__name__, type(a.method).__name__)

---

## `q09`

_q09_

In [ ]:
def q09() -> None:
    # PREDICTION:                         rung:
    a = Base()
    a.method = lambda: "shadowed"       # type: ignore[method-assign]
    print("q09", a.method(), Base.method(a))

---

## `WithGetattr`

_WithGetattr_

In [ ]:
class WithGetattr(Base):
    def __getattr__(self, name: str) -> str:
        return f"<generated {name}>"

---

## `q10`

_q10_

In [ ]:
def q10() -> None:
    # PREDICTION:                         rung:
    w = WithGetattr()
    print("q10", w.own, w.anything_at_all)

---

## `q11`

_q11_

In [ ]:
def q11() -> None:
    # PREDICTION:                         rung:
    w = WithGetattr()
    try:
        print("q11", w.computed)
    except Exception as exc:
        print("q11", type(exc).__name__)

---

## `Mangled`

_Mangled_

In [ ]:
class Mangled:
    def __init__(self) -> None:
        self.__hidden = "secret"

    def reveal(self) -> str:
        return self.__hidden

---

## `q12`

_q12_

In [ ]:
def q12() -> None:
    # PREDICTION: what are the keys of m.__dict__?
    m = Mangled()
    print("q12", list(m.__dict__), m.reveal())
    try:
        print("q12", m.__hidden)          # type: ignore[attr-defined]
    except AttributeError as exc:
        print("q12 AttributeError:", exc)

---

## `Tracer`

Implement __getattribute__ so that EVERY attribute access is logged,

In [ ]:
class Tracer:
    """Implement __getattribute__ so that EVERY attribute access is logged,
    then use it to trace the ladder on a real object.

    Requirements:
      - log the attribute name to self._log
      - delegate to super().__getattribute__ for the actual lookup
      - do NOT infinitely recurse (accessing self._log inside
        __getattribute__ is itself an attribute access -- this is the trap)

    Then answer:
      - which of q01-q12 above would produce a DIFFERENT log if you used
        __getattr__ instead of __getattribute__?
      - why is __getattribute__ almost never the right tool in production code?
    """

    def __init__(self) -> None:
        self._log: list[str] = []
        self.value = 42

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    for fn in [q01, q02, q03, q04, q05, q06, q07, q08, q09, q10, q11, q12]:
        fn()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.